# Hotel Booking Cancellation Prediction

IT3091 Machine Learning Project | Group 6 | Tourism and Hospitality

This project compares models for predicting booking cancellations and groups
guests by booking behaviour. The final examples show how model probabilities
can support deposit recommendations and daily overbooking estimates.

Dataset: Antonio, N., de Almeida, A. and Nunes, L. (2019),
“Hotel booking demand datasets”, Data in Brief, 22, pp. 41–49.

## Notebook contents
1. Project objective
2. Method choices
3. Data exploration
4. Data preparation
5. Model training
6. Test results
7. Decision threshold
8. Prediction explanations
9. Guest segmentation
10. Booking decision example
11. Recommendations and limitations


In [ ]:
# Step 0: Imports and Environment Setup
import os
import sys
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Find the project root whether launched from the project or notebooks folder.
from pathlib import Path
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "models.py").exists()), None)
if project_root is None:
    raise RuntimeError("Open this notebook from the hotel booking project folder.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Set styling
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.family"] = "sans-serif"

# Ensure reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("[INFO] Libraries loaded successfully. Python Version:", sys.version.split()[0])


## 1. Project objective

Hotels lose potential revenue when bookings are cancelled and rooms remain
empty. Accepting too many extra bookings can also leave a hotel without enough
rooms for arriving guests. This project explores how cancellation predictions
could help hotel managers balance these risks.

The dataset contains 119,390 bookings. About 37% were cancelled before cleaning.
Each row describes one booking, and the target is `is_canceled`:

- `1`: the booking was cancelled.
- `0`: the booking was not cancelled.

The main task is to estimate the probability that a booking will be cancelled.
The decision example uses this probability to assign a risk tier and suggest
an action. A separate clustering analysis groups bookings with similar guest
and stay characteristics.

The intended users are hotel revenue managers. Before using the model at a
specific stage of booking, they would need to confirm that every input feature
is available at that time.


## 2. Method choices

| Area | Choice | Reason and interpretation |
|---|---|---|
| Outcome information | Remove `reservation_status` and `reservation_status_date` | These columns reveal the booking outcome. |
| Missing values | Fill missing children counts with 0 and countries with `Unknown`; create agent and company indicators | Preserve usable bookings and represent missing agent or company records explicitly. |
| Invalid records | Remove bookings with no guests and nightly prices outside 0–1,000 | The run removes 182 records. |
| Data split | Stratified 80/20 split, seed 42 | Use 95,366 bookings for training and 23,842 for testing, with similar target proportions. |
| Model comparison | Two baselines and five learned models | Compare simple rules with linear, bagging, boosting and neural models. The learned models use five-fold validation. |
| Evaluation | Accuracy, precision, recall, F1, ROC-AUC, PR-AUC and Brier score | Examine classification performance and the quality of the probability estimates. |
| Operational model | Explicitly select XGBoost | This is a configured choice; Random Forest has the highest test accuracy in this run. |
| Decision threshold | Compare thresholds from 0.05 to 0.95 | Find the highest estimated value under the specified cost assumptions. The result is not evidence of realised hotel profit. |
| Guest groups | Fit four K-Means clusters and inspect elbow and silhouette plots | Summarise booking behaviour and display the groups using PCA. |

Preprocessing is fitted on the training split before cross-validation, and
the decision threshold is selected using test data. These limitations are
discussed at the end of the notebook.


In [ ]:
# Step 1: Data Ingestion & Data Dictionary Retrieval
from src.data_loader import load_raw_data, get_data_dictionary

df_raw = load_raw_data()
print(f"[DataLoader] Ingested raw dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

# Inspect Data Dictionary Sample
data_dict = get_data_dictionary()
print(f"[DataLoader] Registered {len(data_dict)} attributes in Data Dictionary.")
df_raw.head()


## 3. Data exploration

Load the booking data, check missing and invalid values, and inspect patterns
in cancellations, lead time, deposit type, requests, parking and seasonality.
The cells below generate the summary statistics and six charts.


In [ ]:
# Step 2: Statistical Profiling and Visualization
from src.eda_engine import generate_eda_profile, plot_eda_figures

eda_summary = generate_eda_profile(df_raw)
plot_eda_figures(df_raw)

# Display Key EDA Metrics
print("Overall Cancellation Rate:", eda_summary["cancellation_rate_overall"], "%")
print("Resort Hotel Cancellation Rate:", eda_summary["hotel_breakdown"]["Resort Hotel"]["cancel_rate"], "%")
print("City Hotel Cancellation Rate:", eda_summary["hotel_breakdown"]["City Hotel"]["cancel_rate"], "%")
print("Anomalous 0-Guest Records Filtered:", eda_summary["anomalies"]["zero_total_guests"])


In [ ]:
# Display Generated EDA Figures
from IPython.display import Image, display

display(Image(filename="reports/figures/fig1_target_hotel_distribution.png"))
display(Image(filename="reports/figures/fig2_lead_time_distribution.png"))
display(Image(filename="reports/figures/fig3_deposit_market_segment.png"))
display(Image(filename="reports/figures/fig4_special_requests_parking.png"))
display(Image(filename="reports/figures/fig5_monthly_seasonality.png"))
display(Image(filename="reports/figures/fig6_correlation_matrix.png"))


## 4. Data preparation

Clean invalid bookings and handle missing values. Create features for total
stay, guest count, family bookings, room mismatch, price per guest, past
cancellations, season and booking commitment.

Separate the target and make the stratified train/test split. Fit numerical
scaling and categorical encoding on the training data, then apply the fitted
transformations to the test data.


In [ ]:
# Step 3: Leak-Free Preprocessing, Feature Engineering & Partitioning
from src.preprocessing import prepare_train_test_data

data = prepare_train_test_data(df_raw, test_size=0.2, random_state=42)
X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
feature_names = data["feature_names"]
df_featured = data["df_featured"]

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")
print(f"Engineered Feature Space: {len(feature_names)} features")


## 5. Model training

Compare a majority-class baseline, a lead-time rule, Logistic Regression,
Random Forest, XGBoost, LightGBM and a neural network. The five learned models
use five-fold stratified cross-validation and are then fitted on the full
training split. The two baselines are evaluated directly on training data.

The fitted models are stored in `fitted_models` and saved under
`data/processed/models`.


In [ ]:
# Step 4: Model Cross-Validation and Fitting
from src.models import train_and_cross_validate_models

cv_results, fitted_models = train_and_cross_validate_models(X_train, y_train, n_splits=5)

# Convert CV results to DataFrame for structured viewing
df_cv = pd.DataFrame(cv_results).T
df_cv[["cv_roc_auc_mean", "cv_roc_auc_std", "cv_f1_mean", "cv_precision_mean", "cv_recall_mean", "cv_accuracy_mean", "fit_time_seconds"]]


## 6. Test results

Evaluate every model on the held-out test split. The table includes accuracy,
precision, recall, F1, ROC-AUC, PR-AUC and Brier score. The plots compare ranking
performance and how closely predicted probabilities match observed outcomes.


In [ ]:
# Step 5: Test Set Evaluation & Curves
from src.evaluation import evaluate_models_on_test_set, plot_roc_and_pr_curves, plot_calibration_curves

test_metrics, preds_dict = evaluate_models_on_test_set(fitted_models, X_test, y_test)
plot_roc_and_pr_curves(fitted_models, X_test, y_test)
plot_calibration_curves(fitted_models, X_test, y_test)

df_test = pd.DataFrame(test_metrics).T
df_test[["accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc", "brier_score"]]


In [ ]:
display(Image(filename="reports/figures/fig9_roc_pr_curves.png"))
display(Image(filename="reports/figures/fig10_calibration_curves.png"))


## 7. Decision threshold

Use the selected XGBoost model to compare estimated value across decision
thresholds. The example assumes a loss of 150 for a missed cancellation,
a loss of 40 for an incorrect cancellation flag, a gain of 120 for a correctly
flagged cancellation and a gain of 150 for a correctly predicted arrival.
These values are modelling assumptions, expressed in dollars.

The reported improvement compares the selected threshold with 0.50 under
those assumptions. It is a scenario estimate, not measured annual revenue.


In [ ]:
# Step 6: Cost-Benefit Revenue Optimization
from src.evaluation import run_cost_benefit_optimization

champion_model = fitted_models["XGBoost"]
cost_summary = run_cost_benefit_optimization(
    champion_model, X_test, y_test,
    cost_fn=150.0,  # Cost of False Negative (Perishable room night lost)
    cost_fp=40.0,   # Cost of False Positive (Overbooking relocation friction)
    gain_tp=120.0,  # Value of True Positive (Salvaged room revenue via resale)
    gain_tn=150.0   # Value of True Negative (Normal guest check-in revenue)
)

print(f"Optimal Decision Cutoff T*: {cost_summary['optimal_threshold']}")
print(f"Default Cutoff Profit per Booking (T=0.50): ${cost_summary['default_profit_per_booking']}")
print(f"Optimal Cutoff Profit per Booking: ${cost_summary['optimal_profit_per_booking']}")
print(f"Net Profit Lift per Booking: +${cost_summary['profit_lift_per_booking']}")
print(f"Annualized Net Gain per 100k Bookings: +${cost_summary['annualized_gain_per_100k_bookings']:,}")

display(Image(filename="reports/figures/fig11_cost_benefit_optimization.png"))


## 8. Prediction explanations

Calculate SHAP values for a sample of test bookings. The feature-importance
chart summarises the largest contributions to model predictions. The beeswarm
plot shows their direction and spread. These explanations describe the model;
they do not establish that a feature causes cancellations.


In [ ]:
# Step 7: SHAP Global and Local Explainability
from src.explainability import compute_shap_explanations

shap_results = compute_shap_explanations(champion_model, X_test, feature_names, sample_size=1500)

display(Image(filename="reports/figures/fig12_shap_feature_importance.png"))
display(Image(filename="reports/figures/fig13_shap_beeswarm.png"))


## 9. Guest segmentation

Group bookings using lead time, stay duration, guest count, nightly price,
past cancellation ratio, requests, parking and repeated-guest status.
Inspect clustering scores, fit four groups, and summarise each group's booking
characteristics. PCA is used to display the groups in two dimensions.


In [ ]:
# Step 8: K-Means Behavioral Clustering
from src.segmentation import perform_customer_segmentation

personas = perform_customer_segmentation(df_featured, n_clusters=4, sample_size=15000)

for c, p in personas.items():
    print(f"Cluster {c}: {p['persona_name']} ({p['pct']}% of guests)")
    print(f"   Avg Lead Time: {p['lead_time_mean']}d | Avg Stay: {p['stay_nights_mean']} nights | ADR: ${p['adr_mean']} | Cancel Rate: {p['cancellation_rate_pct']}%")
    print(f"   Action: {p['recommended_strategy']}\n")

display(Image(filename="reports/figures/fig7_cluster_elbow_silhouette.png"))
display(Image(filename="reports/figures/fig8_customer_clusters_pca.png"))


## 10. Booking decision example

Score one example reservation using the saved preprocessor and XGBoost model.
Display its cancellation probability, risk tier, deposit recommendation and
suggested action. Then estimate a conservative overbooking buffer for a sample
day with 200 bookings.

Risk tiers use fixed boundaries of 25% and 55%. The separate operational
action trigger uses the threshold calculated in Section 7.


In [ ]:
# Booking risk and daily overbooking example
from src.decision_support import HotelDecisionEngine

engine = HotelDecisionEngine(model_name="xgboost")

sample_reservation = {
    "hotel": "City Hotel",
    "lead_time": 180,
    "arrival_date_year": 2017,
    "arrival_date_month": "August",
    "arrival_date_week_number": 32,
    "arrival_date_day_of_month": 15,
    "stays_in_weekend_nights": 2,
    "stays_in_week_nights": 3,
    "adults": 2,
    "children": 0,
    "babies": 0,
    "meal": "BB",
    "country": "PRT",
    "market_segment": "Online TA",
    "distribution_channel": "TA/TO",
    "is_repeated_guest": 0,
    "previous_cancellations": 1,
    "previous_bookings_not_canceled": 0,
    "reserved_room_type": "A",
    "assigned_room_type": "A",
    "booking_changes": 0,
    "deposit_type": "No Deposit",
    "agent": 9.0,
    "company": None,
    "days_in_waiting_list": 0,
    "customer_type": "Transient",
    "adr": 125.0,
    "required_car_parking_spaces": 0,
    "total_of_special_requests": 0
}

risk = engine.predict_booking_risk(sample_reservation)
print("Real-Time Reservation Risk Evaluation:")
print("   Cancellation Probability:", risk["cancellation_probability_pct"], "%")
print("   Operational Tier:        ", risk["risk_tier"])
print("   Deposit Policy:          ", risk["recommended_deposit_policy"])
print("   Recommended Action:      ", risk["operational_action"])

# Statistical Poisson-Binomial Daily Overbooking Calculation
sample_day_probs = champion_model.predict_proba(X_test[:200])[:, 1]
ob = engine.calculate_overbooking_buffer(sample_day_probs, max_walk_risk_alpha=0.005)
print("\nDaily Overbooking Calculation (N=200 Bookings):")
print(f"   Expected Cancellations: {ob['expected_cancellations_mean']} rooms")
print(f"   Safe Overbooking Buffer: {ob['recommended_safe_overbooking_rooms']} rooms (Confidence: {ob['confidence_level_pct']}%)")


## 11. Recommendations and limitations

### Possible uses
- Review high-risk bookings for reconfirmation or a deposit requirement.
- Send pre-arrival reminders for medium-risk bookings.
- Use aggregated cancellation probabilities to support occupancy planning.
- Review guest groups when considering offers and service arrangements.

These actions are suggestions. Their effect on cancellations and customer
experience has not been tested in this project.

### Limitations
- The data covers two hotels in Portugal from 2015–2017. Results may differ
  for other hotels and for future bookings.
- The split is random rather than chronological. A later-period test would
  better assess predictions for future bookings.
- Country grouping is calculated before the split. Scaling and encoding are
  fitted before cross-validation rather than separately inside each fold.
- The decision threshold is tuned on the test set. A separate validation set
  should be used before reporting final business performance.
- Assigned room type and booking changes may be unavailable when a reservation
  is first created. Input availability needs to match the prediction time.
- Country grouping is recalculated when scoring an individual booking; the
  training mapping should instead be reused for consistent deployment.
- Deposit type and other inputs may reflect existing hotel policies. Their
  association with cancellation does not establish a causal effect.
- The overbooking buffer uses an independence assumption and a normal
  approximation. The stated risk tolerance is not a guaranteed guest walk rate.
- The original web dashboard uses a scoring formula rather than calling the
  trained model. The Python example above does use the saved model.
- No completed demographic fairness audit is demonstrated by this notebook.

### AI assistance
Generative AI assisted with brainstorming, code organisation and checking
parts of the project. This disclosure should be reviewed by the project team
to ensure it accurately describes their use and meets the course requirements.
